In [ ]:
# Install required packages.
# ultralytics provides YOLOv8 and the `yolo` CLI command.
# pyyaml is used to read/write the dataset YAML file.
!pip install ultralytics pyyaml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 8.0 MB/s eta 0:00:00


In [1]:
# Mount Google Drive before using any /content/drive/... paths.
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Custom furniture object-detection training

Your dataset appears to be an object-detection dataset with `images` and `labels` folders, so the correct task is `detect`, not `classify`.

In [ ]:
# Import YAML library to safely edit the data.yaml file.
import yaml

# Path to your object-detection dataset YAML file.
yaml_path = "/content/drive/MyDrive/Furniture_DataSet/data (2).yaml"

# Load the existing YAML file into a Python dictionary.
with open(yaml_path, "r") as f:
    data = yaml.safe_load(f)

# Display the current YAML content before modification.
print("Before edit:", data)

Before edit: {'names': ['Chair', 'Sofa', 'Table'], 'nc': 3, 'test': '/content/drive/MyDrive/Likitha/Furniture_DataSet/test', 'train': '/content/drive/MyDrive/Likitha/Furniture_DataSet/train', 'val': '/content/drive/MyDrive/Likitha/Furniture_DataSet/test'}


In [ ]:
# Update dataset paths and class information.
# YOLO detection datasets normally contain train/images, train/labels, test/images, test/labels, etc.
data["train"] = "/content/drive/MyDrive/Furniture_DataSet/test"
data["val"] = "/content/drive/MyDrive/Furniture_DataSet/test"
data["test"] = "/content/drive/MyDrive/Furniture_DataSet/test"

# Number of classes.
data["nc"] = 3

# Class names. These must match the class IDs used in your label .txt files:
# 0 -> Chair, 1 -> Sofa, 2 -> Table
data["names"] = ["Chair", "Sofa", "Table"]

# Save the corrected YAML file.
with open(yaml_path, "w") as f:
    yaml.dump(data, f, sort_keys=False)

# Print the corrected YAML content.
print("After edit:", data)

After edit: {'names': ['Chair', 'Sofa', 'Table'], 'nc': 3, 'test': '/content/drive/MyDrive/Likitha/Furniture_DataSet/test', 'train': '/content/drive/MyDrive/Likitha/Furniture_DataSet/test', 'val': '/content/drive/MyDrive/Likitha/Furniture_DataSet/test'}


In [ ]:
# Train a YOLOv8 object-detection model on your furniture dataset.
from ultralytics import YOLO

# Load the pretrained nano detection model.
model = YOLO("yolov8n.pt")

# Train for 10 epochs. Increase epochs later for better accuracy.
train_results = model.train(
    data=yaml_path,
    epochs=10,
    imgsz=640
)

Ultralytics 8.4.156 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Likitha/Furniture_DataSet/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, n

In [ ]:
# Find the latest trained detection weights automatically.
# This avoids errors caused by hard-coding train, train2, train3, etc.
from pathlib import Path

runs_dir = Path("/content/runs/detect")

# Collect all best.pt files under /content/runs/detect/*/weights/best.pt.
best_weights = sorted(
    runs_dir.glob("*/weights/best.pt"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

# Stop with a clear error message if no trained weights are found.
if not best_weights:
    raise FileNotFoundError("No best.pt found. Please run the training cell first.")

# Use the newest best.pt file.
best_model_path = str(best_weights[0])
print("Using model:", best_model_path)

Using model: /content/runs/detect/train-2/weights/best.pt


In [ ]:
# Predict on a single image using the trained object-detection model.
# IMPORTANT: model must point to best.pt, not args.yaml.
trained_model = YOLO(best_model_path)

single_image_results = trained_model.predict(
    source="/content/drive/MyDrive/Furniture_DataSet/CHAIR.jpg",
    conf=0.1,
    save=True
)

In [ ]:
# Predict on multiple images from a folder.
# Change this folder path if your test images are somewhere else.
folder_results = trained_model.predict(
    source="/content/drive/MyDrive/Furniture_DataSet/test/images",
    conf=0.1,
    save=True
)

In [ ]:
# Print detection results in a readable format.
for result in folder_results:
    print("File:", result.path)

    # result.boxes contains all detected bounding boxes.
    if result.boxes is None or len(result.boxes) == 0:
        print("No objects detected")
    else:
        for box in result.boxes:
            class_id = int(box.cls[0])
            confidence = float(box.conf[0])
            class_name = result.names[class_id]
            coordinates = box.xyxy[0].tolist()

            print("Class:", class_name)
            print("Confidence:", round(confidence, 4))
            print("Box [x1, y1, x2, y2]:", [round(x, 2) for x in coordinates])

    print("-" * 50)